# NEXUS / GAIA — Multi-Account Drive ↔ Colab Federation v2.2

This notebook performs a full evidence-first scan of the Drive files visible to the authenticated Colab identity, including shared roots configured in Colab Secrets. It discovers historical Colab notebooks and project artifacts, hashes bytes and normalized notebook source, identifies duplicates and lineage, reviews risky code without executing it, writes a hash-chained receipt ledger, and pushes report versions to the Drive outbox.

**Full-scope traversal:** configured folder roots are walked breadth-first with no default depth cap. A depth cap is opt-in only.\n\n**Important boundary:** ChatGPT-connected OAuth sessions are not exportable to Colab. Federation works by sharing each source account's project folders with the active Colab identity or placing files in the shared bridge folder. No imported notebook or shell script is automatically executed.

In [ ]:
%pip -q install google-api-python-client requests

In [ ]:
from pathlib import Path
import hashlib, importlib.util, json, os, requests, sys, time

WORKSPACE = Path("/content/nexus_bridge")
WORKSPACE.mkdir(parents=True, exist_ok=True)
MODULE_REF = "03f52a841635d0f6d23a2f8d0fdb6f54298120e8"
MODULE_URL = f"https://raw.githubusercontent.com/machackabook/Nexus_Core/{MODULE_REF}/gaia/nexus/drive_federation_v2.py"
module_response = requests.get(MODULE_URL, timeout=45)
module_response.raise_for_status()
module_path = WORKSPACE / "drive_federation_v2.py"
module_path.write_bytes(module_response.content)
MODULE_SHA256 = hashlib.sha256(module_response.content).hexdigest()
spec = importlib.util.spec_from_file_location("drive_federation_v2", module_path)
federation_module = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = federation_module
spec.loader.exec_module(federation_module)
DriveFederation = federation_module.DriveFederation
HashChain = federation_module.HashChain
print("Bridge module loaded:", federation_module.SCHEMA_VERSION, MODULE_SHA256)

In [ ]:
from google.colab import auth, userdata
import google.auth
from googleapiclient.discovery import build

auth.authenticate_user()
credentials, _ = google.auth.default()
drive = build("drive", "v3", credentials=credentials, cache_discovery=False)
about = drive.about().get(fields="user(displayName,emailAddress),storageQuota").execute()
active_email = str(about.get("user", {}).get("emailAddress", ""))
active_alias = hashlib.sha256(active_email.lower().encode("utf-8")).hexdigest()[:12] if active_email else "unknown"
print("Authenticated Colab principal alias:", active_alias)
print("Email intentionally not printed or exported.")

In [ ]:
def read_colab_secret(name, default=""):
    try:
        value = userdata.get(name)
        return value if value is not None else default
    except Exception:
        return default

raw_config = read_colab_secret("NEXUS_DRIVE_BRIDGE_CONFIG_JSON", "{}")
CONFIG = json.loads(raw_config or "{}")
if not isinstance(CONFIG, dict):
    raise TypeError("NEXUS_DRIVE_BRIDGE_CONFIG_JSON must decode to an object")
ROOTS = CONFIG.get("source_roots", [])
if not isinstance(ROOTS, list):
    raise TypeError("source_roots must be a JSON list")
ROOTS = [
    {"label": str(root.get("label", "shared-root")), "folder_id": str(root.get("folder_id", "")).strip()}
    for root in ROOTS
    if isinstance(root, dict) and str(root.get("folder_id", "")).strip()
]
MAX_INSPECT_FILES = CONFIG.get("max_inspect_files")  # null = inspect every candidate
MAX_DOWNLOAD_BYTES = int(CONFIG.get("max_download_bytes", 50_000_000))
if MAX_INSPECT_FILES is not None:
    MAX_INSPECT_FILES = int(MAX_INSPECT_FILES)
EXPECTED_CONNECTED_IDENTITIES = 9  # historical ChatGPT connector count; not nine Colab OAuth sessions
federation = DriveFederation(drive, WORKSPACE, max_download_bytes=MAX_DOWNLOAD_BYTES)
print("Configured shared source roots:", len(ROOTS))
print("Candidate inspection limit:", MAX_INSPECT_FILES if MAX_INSPECT_FILES is not None else "NONE — all candidates")\nprint("Configured-root traversal:", "UNBOUNDED breadth-first by default")
print("Per-file download ceiling:", MAX_DOWNLOAD_BYTES)
print("Historical connected Drive identities:", EXPECTED_CONNECTED_IDENTITIES)
print("Colab API principal sessions active:", 1)
print("Federation mode: shared-folder permissions + one active Colab principal")

In [ ]:
def find_folder(name, parent_id=None):
    escaped = name.replace("'", "\\'")
    query = "name = '%s' and mimeType = 'application/vnd.google-apps.folder' and trashed = false" % escaped
    if parent_id:
        query += " and '%s' in parents" % parent_id
    result = drive.files().list(
        q=query, pageSize=100, fields="files(id,name,parents,webViewLink,capabilities(canEdit,canAddChildren))",
        includeItemsFromAllDrives=True, supportsAllDrives=True
    ).execute()
    return result.get("files", [])

def create_folder(name, parent_id=None):
    body = {"name": name, "mimeType": "application/vnd.google-apps.folder"}
    if parent_id:
        body["parents"] = [parent_id]
    return drive.files().create(body=body, fields="id,name,parents").execute()

bridge_id = CONFIG.get("bridge_folder_id")
bridge_error = None
try:
    if not bridge_id:
        found = find_folder("NEXUS_MULTIACCOUNT_BRIDGE")
        if found:
            bridge_id = found[0]["id"]
        else:
            colab_roots = find_folder("Colab Notebooks")
            parent = colab_roots[0]["id"] if colab_roots else None
            bridge_id = create_folder("NEXUS_MULTIACCOUNT_BRIDGE", parent).get("id")
except Exception as exc:
    bridge_error = "%s: %s" % (type(exc).__name__, str(exc)[:180])
    bridge_id = None

SUBFOLDERS = {}
if bridge_id:
    for name in ("inbox", "staging", "canonical", "outbox", "receipts", "quarantine"):
        try:
            found = find_folder(name, bridge_id)
            SUBFOLDERS[name] = found[0]["id"] if found else create_folder(name, bridge_id)["id"]
        except Exception as exc:
            SUBFOLDERS[name] = None
            print("SUBFOLDER_UNAVAILABLE", name, type(exc).__name__, str(exc)[:120])

OUTBOX_ID = SUBFOLDERS.get("outbox")
QUARANTINE_ID = SUBFOLDERS.get("quarantine")
print("Bridge folder accessible:", bool(bridge_id))
print("Writable outbox available:", bool(OUTBOX_ID))
if bridge_error:
    print("Bridge folder error:", bridge_error)
print("Bridge subfolders:", {k: bool(v) for k, v in SUBFOLDERS.items()})

In [ ]:
SEARCH_TERMS = federation_module.DEFAULT_TERMS
report = federation.discover(
    terms=SEARCH_TERMS,
    root_ids=ROOTS,
    max_inspect_files=MAX_INSPECT_FILES,
)
report["runtime"] = {
    "active_principal_alias": active_alias,
    "module_version": federation_module.SCHEMA_VERSION,
    "module_sha256": MODULE_SHA256,
    "bridge_folder_configured": bool(bridge_id),
    "expected_chatgpt_connected_identity_count_historical": EXPECTED_CONNECTED_IDENTITIES,
    "colab_oauth_sessions_active": 1,
    "configured_shared_roots": len(ROOTS),
    "candidate_inspection_limit": MAX_INSPECT_FILES,
    "download_ceiling_bytes": MAX_DOWNLOAD_BYTES,
}
report_paths = federation.write_report(report)
scope = report["scope"]
print("Visible Drive objects:", scope["visible_metadata_count"])
print("Relevant candidate artifacts:", scope["candidate_count"])
print("Inspected artifacts:", scope["inspected_count"])
print("Colab/IPYNB notebooks:", scope["colab_notebook_count"])
print("Drive createdTime Jan-Jun 2025 notebooks:", scope["jan_jun_2025_colab_count"])
print("Distinct owner aliases in inspected corpus:", scope["distinct_owner_alias_count"])
print("Configured shared roots:", scope["configured_roots"])
print("User-corpus metadata complete:", scope["metadata_scan_complete_for_user_corpus"])
print("Shared-drive metadata complete:", scope["metadata_scan_complete_for_all_drives"])
print("Inspection limit reached:", scope["inspection_limit_reached"])
print("Exact duplicate groups:", len(report["duplicates"]["exact"]))
print("Normalized duplicate groups:", len(report["duplicates"]["normalized"]))
print("Execution review:", report["execution_review_summary"])
print("Earliest Drive notebook createdTime:", scope["earliest_colab_created_time"])

## Historical target review

The discovery pass looks for Great Assimilation, Cloud Purge, Phi/Phil, coherence, The Nexus Generation, Gemini, GAIA, ADAM, SYNAPSE, Chimera, sparsebundle, and all accessible Colab/IPYNB files.

The date field is Drive createdTime, not proof of original authorship. Imported scripts are inspected as text only. Destructive or credential-handling patterns are reported, never executed.

In [ ]:
# Upload each report as a new Drive object. Existing source artifacts are never overwritten by this default path.
uploaded_reports = []
if OUTBOX_ID:
    for key, path in report_paths.items():
        stamp = time.strftime("%Y%m%d-%H%M%S", time.gmtime())
        name = "%s-%s-%s.%s" % (key, stamp, hashlib.sha256(path.read_bytes()).hexdigest()[:10], path.suffix.lstrip("."))
        try:
            result = federation.upload_new_version(path, OUTBOX_ID, name=name)
            uploaded_reports.append({"kind": key, "id": result.get("id"), "name": result.get("name")})
        except Exception as exc:
            print("REPORT_PUSH_FAILED", key, type(exc).__name__, str(exc)[:180])
else:
    print("DRIVE_OUTBOX_UNAVAILABLE: reports remain in the Colab workspace; no Drive write was attempted.")
print("Drive outbox report versions created:", len(uploaded_reports))
for item in uploaded_reports:
    print(item["kind"], item["name"], item["id"])

In [ ]:
def pull_artifact(file_id, filename=None):
    return federation.pull_file(file_id, filename)

def push_new_version(local_path, name=None):
    if not OUTBOX_ID:
        raise PermissionError("No writable Drive outbox is available to the active Colab identity.")
    return federation.upload_new_version(local_path, OUTBOX_ID, name=name)

def replace_original_with_verified_backup(file_id, local_path, expected_current_sha256):
    if not QUARANTINE_ID:
        raise PermissionError("No writable backup/quarantine folder is available.")
    return federation.replace_original(
        file_id=file_id,
        local_path=local_path,
        expected_current_sha256=expected_current_sha256,
        backup_folder_id=QUARANTINE_ID,
        confirm=True,
    )

print("Bidirectional API ready:")
print("  Drive → Colab: pull_artifact(file_id)")
print("  Colab → Drive: push_new_version(local_path)")
print("  Guarded original replacement: replace_original_with_verified_backup(...)")
print("Default write mode is new-version only; original replacement is never automatic.")

## Gemini Flash backend

The Gemini credential is read only from the Colab Secret named GEMINI_API_KEY. The backend stores it in an ephemeral runtime config file with owner-only permissions. It is not printed, committed, written into the registry, or uploaded to Drive.

If the key pasted into the earlier chat was real, rotate it and place the replacement in Colab Secrets before running the live smoke test.

In [ ]:
gemini_key = read_colab_secret("GEMINI_API_KEY")
gemini_model = read_colab_secret("GEMINI_MODEL", "gemini-flash-latest")
backend_dir = Path("/root/nexus/bin")
config_dir = Path("/root/.config/nexus")
backend_dir.mkdir(parents=True, exist_ok=True)
config_dir.mkdir(parents=True, exist_ok=True)
backend_config = config_dir / "gemini.json"
backend_path = backend_dir / "gemini-backend"

backend_source = r'''#!/usr/bin/env python3
import json, os, sys, urllib.request, urllib.error
from pathlib import Path
config_path = Path("/root/.config/nexus/gemini.json")
config = json.loads(config_path.read_text()) if config_path.exists() else {}
key = os.environ.get("GEMINI_API_KEY") or config.get("GEMINI_API_KEY", "")
model = os.environ.get("GEMINI_MODEL") or config.get("GEMINI_MODEL", "gemini-flash-latest")
if not key:
    print(json.dumps({"error": "GEMINI_API_KEY is not configured"}), file=sys.stderr)
    raise SystemExit(78)
prompt = sys.stdin.read().strip() if not sys.argv[1:] else " ".join(sys.argv[1:]).strip()
if not prompt:
    print(json.dumps({"error": "empty_prompt"}), file=sys.stderr)
    raise SystemExit(64)
url = "https://generativelanguage.googleapis.com/v1beta/models/" + model + ":generateContent"
body = json.dumps({"contents": [{"parts": [{"text": prompt}]}]}).encode("utf-8")
request = urllib.request.Request(
    url, data=body, method="POST",
    headers={"Content-Type": "application/json", "X-goog-api-key": key}
)
try:
    with urllib.request.urlopen(request, timeout=120) as response:
        sys.stdout.write(response.read().decode("utf-8"))
except urllib.error.HTTPError as exc:
    sys.stderr.write(exc.read().decode("utf-8", errors="replace"))
    raise SystemExit(70)
'''
if gemini_key:
    backend_config.write_text(json.dumps({"GEMINI_API_KEY": gemini_key, "GEMINI_MODEL": gemini_model}))
    os.chmod(backend_config, 0o600)
    backend_path.write_text(backend_source)
    os.chmod(backend_path, 0o700)
    print("Gemini backend installed:", backend_path)
    print("Secret file permissions:", oct(backend_config.stat().st_mode & 0o777))
else:
    print("Gemini backend not installed: Colab Secret GEMINI_API_KEY is not configured.")

In [ ]:
if gemini_key:
    import subprocess
    smoke = subprocess.run(
        [str(backend_path)],
        input="Return exactly: NEXUS_COLAB_GEMINI_OK",
        text=True, capture_output=True, timeout=150,
    )
    print("Gemini backend exit code:", smoke.returncode)
    if smoke.returncode != 0:
        print("Backend error:", smoke.stderr[:1200])
        raise RuntimeError("Gemini smoke test failed")
    response_json = json.loads(smoke.stdout)
    response_text = "".join(
        part.get("text", "")
        for part in response_json.get("candidates", [{}])[0].get("content", {}).get("parts", [])
        if isinstance(part, dict)
    ).strip()
    print("Gemini model:", gemini_model)
    print("Gemini response SHA-256:", hashlib.sha256(smoke.stdout.encode("utf-8")).hexdigest())
    print("Gemini response:", response_text)
    assert "NEXUS_COLAB_GEMINI_OK" in response_text
else:
    print("LIVE_GEMINI_TEST=NOT_RUN (missing Colab Secret)")

In [ ]:
receipt_ok, receipt_errors = federation.receipts.verify()
print("Receipt chain:", "VALID" if receipt_ok else "INVALID")
if receipt_errors:
    print("\n".join(receipt_errors))
assert receipt_ok, "Receipt chain verification failed"
print("NEXUS_DRIVE_COLAB_BRIDGE=COMPLETE_FOR_CURRENTLY_VISIBLE_CORPUS")
print("Cross-account coverage is limited to folders shared with the active Colab principal.")

## SSH client contract

    Host colab
      User root
      ProxyCommand colab ssh --proxy-mode -s colab
      StrictHostKeyChecking no
      UserKnownHostsFile /dev/null
      RequestTTY no
      RemoteCommand cd /content 2>/dev/null && exec /root/nexus/bin/gemini-backend

The backend is installed in this Colab runtime. Runtime reset removes the ephemeral installation; re-running this notebook reinstalls it from Colab Secrets.